# JORINOVA NEXUS — Clinical interpretation (Track 4)

Fine-tunes `microsoft/Phi-3-mini-4k-instruct` on the **Claude-labelled** clinical dataset to produce 2–3 sentence interpretations. Goal: offline-capable clinical interpretation so the lab keeps working when internet drops.

## Before pressing Run All

1. **Runtime → Change runtime type → T4 GPU** (free tier is enough)
2. **Upload** `datasets/clinical_with_interp.jsonl` to the Colab files panel
   (this is the output of `scripts/generate_interpretations_with_claude.py`)

## What it does

1. Loads the 43 labelled rows, splits 80/20 train/eval
2. Wraps Phi-3 with LoRA (rank 16 — generative, not classification)
3. Trains 5 epochs at 2e-4 with cosine schedule (~25 min on T4)
4. Scores ROUGE-L vs Claude reference; prints 3 sample predictions
5. Exports an Ollama Modelfile + merged weights you download to your laptop


## 1. Install + import

In [ ]:
!pip install -q transformers==4.45.0 peft==0.13.0 datasets==3.0.0 accelerate==1.0.0 bitsandbytes==0.44.0 trl==0.11.0 rouge-score==0.1.2

In [ ]:
import json, random, torch, statistics
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

random.seed(42); torch.manual_seed(42)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Load the Claude-labelled clinical data

In [ ]:
DATA_PATH = 'clinical_with_interp.jsonl'  # uploaded next to notebook

rows = [json.loads(l) for l in Path(DATA_PATH).read_text(encoding='utf-8').splitlines() if l.strip()]
print(f'loaded {len(rows)} rows')
print('sample row:')
print(json.dumps(rows[0], indent=2)[:500])

## 3. Format as chat-completion supervised pairs

Same SYSTEM prompt as the Claude labeller. The fine-tune learns to mimic Claude's style + guardrails (2–3 sentences, no hallucinated values, suggest ONE next step).

In [ ]:
SYSTEM = (
    'You are a senior consultant pathologist writing the Clinical '
    'Interpretation section of a Rwandan hospital lab report. Write 2-3 '
    'short sentences (30-60 words). State if the panel is normal, '
    'borderline, or abnormal. If abnormal, name the most likely pattern in '
    'plain terms. Suggest ONE concrete next step. Do not invent values. '
    'Do not prescribe meds. Plain prose, no markdown.'
)

def to_chat(r):
    res = r.get('results', [])
    lines = []
    for x in res:
        lines.append(
            f"  - {x.get('test_name','?')} ({x.get('test_code','?')}): "
            f"{x.get('value','?')} {x.get('unit','')} "
            f"[flag {x.get('flag','N')}, ref {x.get('reference','--')}]"
        )
    user = (
        f"Clinical indication: {r.get('diagnosis','(none)')}\n\n"
        f"Lab results:\n" + '\n'.join(lines) + '\n\n'
        f"Write the 2-3 sentence clinical interpretation."
    )
    return {
        'messages': [
            {'role': 'system',    'content': SYSTEM},
            {'role': 'user',      'content': user},
            {'role': 'assistant', 'content': r['interpretation']},
        ],
    }

random.shuffle(rows)
split = int(0.8 * len(rows))
train_ds = Dataset.from_list([to_chat(r) for r in rows[:split]])
eval_ds  = Dataset.from_list([to_chat(r) for r in rows[split:]])
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')

## 4. Load Phi-3 (4-bit quantised) + attach LoRA

In [ ]:
MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

# Higher rank than intent (16 vs 8) - we generate prose, not pick one of 11 labels
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 5. Train

More epochs than intent (5 vs 2) because we have fewer rows and need the model to learn the prose style and the guardrails. Watch the eval loss — if it stops improving after epoch 3, stop early.

In [ ]:
cfg = SFTConfig(
    output_dir='./out',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy='epoch',
    eval_strategy='epoch',
    max_seq_length=1024,
    optim='paged_adamw_8bit',
    bf16=True,
    report_to='none',
)
trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=train_ds, eval_dataset=eval_ds,
    tokenizer=tokenizer,
)
trainer.train()

## 6. Score against the eval set (ROUGE-L vs Claude reference)

ROUGE-L measures longest common subsequence overlap with the Claude reference. Target ≥ 0.4 mean — above that, the model has clearly learned the style. Aim for ≥ 0.5 with more data later.

In [ ]:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def predict(row):
    msgs = row['messages'][:2]  # system + user only
    inp = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    out = model.generate(
        inp, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()

scores = []
for row in eval_ds:
    ref  = row['messages'][2]['content']
    pred = predict(row)
    scores.append(scorer.score(ref, pred)['rougeL'].fmeasure)

print(f'eval ROUGE-L mean   : {statistics.mean(scores):.3f}')
print(f'eval ROUGE-L median : {statistics.median(scores):.3f}')
print()
print('SAMPLE PREDICTIONS')
for row in eval_ds.select(range(min(3, len(eval_ds)))):
    print('\n--- REFERENCE (Claude) ---')
    print(row['messages'][2]['content'])
    print('--- MODEL ---')
    print(predict(row))

## 7. Export as Ollama Modelfile

Merges LoRA into the base, saves a HF model dir, and writes a Modelfile. After downloading both, on your laptop:
```bash
python llama.cpp/convert_hf_to_gguf.py jorinova-clinical-merged \
    --outfile jorinova-clinical.gguf
ollama create jorinova-clinical -f Modelfile
```

In [ ]:
merged = model.merge_and_unload()
merged.save_pretrained('./jorinova-clinical-merged')
tokenizer.save_pretrained('./jorinova-clinical-merged')

MODELFILE = '''FROM ./jorinova-clinical.gguf
PARAMETER num_ctx 2048
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
SYSTEM You are a senior consultant pathologist writing the Clinical Interpretation section of a Rwandan hospital lab report. Write 2-3 short sentences (30-60 words). State if the panel is normal, borderline, or abnormal. If abnormal, name the most likely pattern in plain terms. Suggest ONE concrete next step. Do not invent values. Do not prescribe meds. Plain prose, no markdown.
'''
Path('Modelfile').write_text(MODELFILE)
print('Wrote ./jorinova-clinical-merged/ and Modelfile')
print()
print('Next on your laptop:')
print('  1. Download jorinova-clinical-merged/ from the Colab files panel')
print('  2. Convert to GGUF (llama.cpp/convert_hf_to_gguf.py)')
print('  3. ollama create jorinova-clinical -f Modelfile')
print('  4. Set OLLAMA_MODEL_INTERPRET=jorinova-clinical in backend/.env')